# Korean Stops Production Data Preprocessing

This notebook preprocesses the production data for Korean stops analysis.

## Steps:
1. Load raw acoustic measurements
2. Clean and filter data
3. Merge consonant-vowel rows into syllables
4. Extract participant IDs
5. Merge demographic information
6. Generate derived variables
7. Merge word frequency data
8. Save preprocessed dataset

In [ ]:
# Import libraries
from pathlib import Path
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

## 1. Load Raw Data

In [ ]:
# Load acoustic measurements from vot_f0_results.txt
base_dir = Path.cwd().parent.parent / "data"
file_path = base_dir / "vot_f0_results.txt"

df = pd.read_csv(
    file_path,
    sep="\t",
    na_values=["--undefined--"]
)

print(f"Loaded {len(df)} rows from {file_path}")
print(f"Columns: {list(df.columns)}")

## 2. Initial Data Cleaning

In [ ]:
# Remove unwanted labels
labels_to_remove = ["pp prevoicing", "tt prevoicing", "prevoicing", "oo"]
for label in labels_to_remove:
    df = df[df["label"] != label].reset_index(drop=True)

print(f"After removing unwanted labels: {len(df)} rows")

In [ ]:
# Map edge cases to standard consonant labels
edge_case_map = {
    "ph fricative-like start": "ph",
    "po": "p",
    "pp?": "pp",
    "ph?": "ph",
    "k?": "k",
    "tt?": "tt",
    "kh?": "kh",
    "kk?": "kk",
    "pp? no VOT?": "pp",
    "k fricative-like": "k",
    "k released": "k",
    "kh fricative-like": "kh",
    "p voiced": "p",
    "ok?": "ok",
}
df["label"] = df["label"].replace(edge_case_map)

print(f"Mapped {len(edge_case_map)} edge cases to standard labels")

In [ ]:
# Convert numeric columns to float
numeric_cols = ["VOT", "f0", "f0_5ms_after_onset", "F1_midpoint", "F2_midpoint"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Numeric columns converted successfully")

## 3. Merge Rows into Syllables
Combine consonant-vowel-coda rows into single syllable entries using sliding window approach.

In [ ]:
# Define valid consonants and Korean word list
valid_consonants = {"t", "p", "k", "tt", "pp", "kk", "kh", "th", "ph"}
unique_korean_words_romanized = [
    'pi', 'nan', 'nos', 'os', 'ppok', 'ssi', 'tan', 'mam', 'kak', 'mang', 'kkong', 'ppi', 
    'cas', 'kkak', 'hak', 'khok', 'ttok', 'pok', 'sol', 'nal', 'chak', 'khan', 'mom', 
    'ttong', 'thong', 'chok', 'man', 'ti', 'am', 'khi', 'an', 'son', 'ak', 'nom', 'al', 
    'nok', 'nak', 'tok', 'nas', 'mal', 'kan', 'com', 'hon', 'chon', 'ttak', 'thak', 'hok', 
    'tti', 'i', 'phi', 'thi', 'phak', 'khak', 'nong', 'ki', 'kim', 'phan', 'nap', 'si', 
    'thok', 'mas', 'kkok', 'pan', 'ppong', 'ok', 'mos', 'than', 'cim', 'kkan', 'tong', 
    'phong', 'som', 'ttan', 'tak', 'pong', 'kok', 'mak', 'phok', 'kki', 'mi', 'sok', 'non', 
    'mok', 'chim', 'nam', 'him', 'nim', 'pak', 'ppan', 'ppak', 'soth', 'aph', 'kong'
]

# Sliding window merge: handle 3-row and 2-row combinations
merged_rows = []
i = 0
while i < len(df):
    # Try 3-row combination first
    if i + 2 < len(df):
        row1, row2, row3 = df.iloc[i], df.iloc[i+1], df.iloc[i+2]
        combined = row1["label"] + row2["label"] + row3["label"]
        
        if (row1["filename"] == row2["filename"] == row3["filename"] and 
            combined in unique_korean_words_romanized):
            merged_rows.append({
                "filename": row1["filename"],
                "consonant": row1["label"],
                "rest": row2["label"] + row3["label"],
                "f0": row2.get("f0_5ms_plus_more_after_onset", None),
                "VOT": row1["duration"] * 1000,  # Convert s to ms
                "rest_duration": (row2["duration"] + row3["duration"]) * 1000
            })
            i += 3
            continue
    
    # Try 2-row combination
    if i + 1 < len(df):
        row1, row2 = df.iloc[i], df.iloc[i+1]
        if row1["filename"] == row2["filename"]:
            merged_rows.append({
                "filename": row1["filename"],
                "consonant": row1["label"],
                "rest": row2["label"],
                "f0": row2.get("f0_5ms_plus_more_after_onset", None),
                "VOT": row1["duration"] * 1000,
                "rest_duration": row2["duration"] * 1000
            })
            i += 2
            continue
    
    i += 1

merged_df = pd.DataFrame(merged_rows)

# Create item column (consonant + rest, removing special characters)
merged_df['item'] = (merged_df['consonant'] + merged_df['rest']).str.replace(
    r'[^a-zA-Z가-힣0-9]', '', regex=True
)

# Validate consonants
invalid_rows = merged_df[~merged_df["consonant"].isin(valid_consonants)]
if len(invalid_rows) > 0:
    print(f"Warning: {len(invalid_rows)} rows with invalid consonants")

print(f"Merged into {len(merged_df)} syllable entries")

## 4. Extract Participant IDs

In [ ]:
def extract_ids(filename):
    """Extract prolific_id and session_number from filename"""
    # Format: proliferate_id-worker_id-session_number or proliferate_id-session_number
    m = re.match(r"^(\d+)-(\d+)-(\d+)", str(filename))
    if m:
        return m.group(1), m.group(3)
    
    m = re.match(r"^(\d+)-(\d+)", str(filename))
    if m:
        return m.group(1), m.group(2)
    
    return None, None

# Extract IDs
merged_df['prolific_id'], merged_df['session_number'] = zip(
    *merged_df['filename'].map(extract_ids)
)

print(f"Extracted IDs for {merged_df['prolific_id'].notna().sum()} participants")

## 5. Merge Demographic Information

In [ ]:
# Load perception data for demographic information
perception_subject_path = base_dir / "korean_stops_perception_3_poa_all_ages-subject_information.csv"
perception_workerids_path = base_dir / "korean_stops_perception_3_poa_all_ages-workerids.csv"

perception_subject_df = pd.read_csv(perception_subject_path, dtype=str)
perception_workerids_df = pd.read_csv(perception_workerids_path, dtype=str)

# Create workerid to prolific_id mapping
workerid_to_prolific = dict(zip(
    perception_workerids_df['workerid'], 
    perception_workerids_df['prolific_participant_id']
))

# Create prolific_id to demographic info mapping
prolific_to_age = {}
prolific_to_gender = {}

for idx, row in perception_subject_df.iterrows():
    workerid = row['workerid']
    age = row['age']
    gender = row.get('gender', None)
    prolific_id = workerid_to_prolific.get(workerid, None)
    
    if prolific_id is not None:
        prolific_to_age[prolific_id] = age
        prolific_to_gender[prolific_id] = gender

print(f"Loaded demographic data for {len(prolific_to_age)} participants")

In [ ]:
merged_df

In [ ]:
# Merge demographic information
ages = []
genders = []
missing_age_count = 0

for idx, row in merged_df.iterrows():
    prolific_id = str(row['prolific_id'])
    age = prolific_to_age.get(prolific_id, None)
    gender = prolific_to_gender.get(prolific_id, None)
    
    # Post-process age values
    if age == '196707':
        age = str(2025 - 1967)
    elif age == '-59':
        age = '59'
    
    # Estimate age from prolific_id if missing
    if age is None and pd.notnull(row['prolific_id']):
        missing_age_count += 1
        try:
            decade = int(str(row['prolific_id'])[0])
            if 2 <= decade <= 9:
                age = str(decade * 10 + 5)
        except Exception:
            age = None
    
    ages.append(age)
    genders.append(gender)

merged_df['age'] = pd.to_numeric(ages, errors='coerce')
merged_df['gender'] = genders

# Fill missing gender values with 'Female' (based on original analysis)
merged_df['gender'] = merged_df['gender'].fillna('Female')

print(f"Merged demographic data")
print(f"  Ages available: {merged_df['age'].notna().sum()}/{len(merged_df)}")
print(f"  Genders available: {merged_df['gender'].notna().sum()}/{len(merged_df)}")
print(f"  Estimated ages for {missing_age_count} participants")

## 6. Generate Derived Variables
Filter for stops only and create analysis variables.

In [ ]:
# Rename for consistency
stops_df = merged_df.rename(columns={'consonant': 'label', 'VOT': 'vot'}).copy()

# Convert to numeric
stops_df['f0'] = pd.to_numeric(stops_df['f0'], errors='coerce')
stops_df['vot'] = pd.to_numeric(stops_df['vot'], errors='coerce')
stops_df['age'] = pd.to_numeric(stops_df['age'], errors='coerce')

# Filter out vowels (keep only stops)
stops_df = stops_df[~stops_df['label'].isin(['a', 'i', 'o'])].copy()

print(f"Filtered to {len(stops_df)} stop consonant tokens")

In [ ]:
# Participant-level normalized variables (z-scores within participant)
stops_df['normed_f0'] = stops_df.groupby('prolific_id')['f0'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0
)
stops_df['normed_vot'] = stops_df.groupby('prolific_id')['vot'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0
)

print("Created participant-normalized variables")

In [ ]:
# Duration variables
stops_df['word_duration'] = stops_df['vot'] + stops_df['rest_duration']
stops_df['scaled_vot_by_rest_duration'] = stops_df['vot'] / stops_df['rest_duration']
stops_df['scaled_vot'] = stops_df['vot'] / stops_df['word_duration']

# Normalized duration variables
stops_df['normed_scaled_vot'] = stops_df.groupby('prolific_id')['scaled_vot'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0
)
stops_df['normed_word_duration'] = (
    (stops_df['word_duration'] - stops_df['word_duration'].mean()) / 
    stops_df['word_duration'].std()
)
stops_df['normed_rest_duration'] = (
    (stops_df['rest_duration'] - stops_df['rest_duration'].mean()) / 
    stops_df['rest_duration'].std()
)

# Log-transformed VOT
stops_df['log_vot'] = stops_df['vot'].apply(lambda x: np.log(x) if x > 0 else np.nan)

print("Created duration and scaled variables")

In [ ]:
# Phonological features
stops_df['vowel'] = stops_df['rest'].str[0]

# Place of articulation (POA)
poa_map = {
    'k': 'dorsal', 'kh': 'dorsal', 'kk': 'dorsal',
    't': 'coronal', 'th': 'coronal', 'tt': 'coronal',
    'p': 'labial', 'ph': 'labial', 'pp': 'labial'
}
stops_df['poa'] = stops_df['label'].map(poa_map)

# Phonation type
phonation_map = {
    'k': 'lenis', 'kk': 'fortis', 'kh': 'aspirated',
    't': 'lenis', 'tt': 'fortis', 'th': 'aspirated',
    'p': 'lenis', 'pp': 'fortis', 'ph': 'aspirated',
}
stops_df['phonation'] = stops_df['label'].map(phonation_map)

print("Created phonological feature variables")

In [ ]:
# Syllable structure
def get_syllable_structure(rest):
    """Determine syllable structure from rest of word after consonant"""
    if pd.isna(rest):
        return pd.NA
    
    rest_str = str(rest)
    
    # Check endings
    if rest_str.endswith(('ng', 'n', 'm')):
        return 'CVN'
    elif rest_str.endswith(('#', 'a', 'i', 'e', 'o', 'u')):
        return 'CV'
    elif rest_str.endswith(('s', 'p', 't', 'k')):
        return 'CVC'
    else:
        return pd.NA

stops_df['syllable_structure'] = stops_df['rest'].apply(get_syllable_structure)

print("Created syllable structure variable")
print(stops_df['syllable_structure'].value_counts(dropna=False))

In [ ]:
# Global normalized age
stops_df['normed_age'] = (
    (stops_df['age'] - stops_df['age'].mean()) / stops_df['age'].std()
)

print("Created normalized age variable")

In [ ]:
# Age group categorical variables
stops_df['age_group'] = pd.cut(
    stops_df['age'], 
    bins=[0, 39, 59, 100], 
    labels=['20s-30s', '40s-50s', '60+']
)
stops_df['detailed_age_group'] = pd.cut(
    stops_df['age'], 
    bins=[0, 29, 39, 49, 59, 100], 
    labels=['20s', '30s', '40s', '50s', '60+']
)

print("Created age group variables")
print(f"\nAge group distribution:")
print(stops_df['age_group'].value_counts().sort_index())
print(f"\nDetailed age group distribution:")
print(stops_df['detailed_age_group'].value_counts().sort_index())

## 7. Merge Word Frequency Data (Optional)

In [ ]:
# Try to load frequency data
freq_file_path = base_dir / 'target_words_frequency.csv'

if freq_file_path.exists():
    freq_df = pd.read_csv(freq_file_path, encoding='utf-8-sig')
    
    # Process frequency data
    freq_df['morpheme_freq_plus1'] = freq_df['morpheme_freq'] + 1
    freq_df['syllable_freq_plus1'] = freq_df['syllable_freq'] + 1
    freq_df['log_morpheme_freq'] = np.log(freq_df['morpheme_freq_plus1'])
    freq_df['log_syllable_freq'] = np.log(freq_df['syllable_freq_plus1'])
    
    # Z-score normalize
    freq_df['z_log_morpheme_freq'] = (
        (freq_df['log_morpheme_freq'] - freq_df['log_morpheme_freq'].mean()) / 
        freq_df['log_morpheme_freq'].std()
    )
    freq_df['z_log_syllable_freq'] = (
        (freq_df['log_syllable_freq'] - freq_df['log_syllable_freq'].mean()) / 
        freq_df['log_syllable_freq'].std()
    )
    
    # Merge with stops_df
    freq_cols = freq_df[['romanization', 'log_morpheme_freq', 'log_syllable_freq', 
                          'z_log_morpheme_freq', 'z_log_syllable_freq']].copy()
    freq_cols = freq_cols.rename(columns={'romanization': 'item'})
    stops_df = stops_df.merge(freq_cols, on='item', how='left')
    
    print(f"Merged frequency data from {freq_file_path}")
    print(f"  Missing frequency data: {stops_df['log_morpheme_freq'].isna().sum()} items")
else:
    print(f"Frequency file not found at {freq_file_path}")
    print("Skipping frequency data merge")

## 8. Save Preprocessed Data

In [ ]:
# Save preprocessed data
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "production_preprocessed_data.csv"

stops_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n{'='*60}")
print("Preprocessing Complete!")
print(f"{'='*60}")
print(f"Final dataset: {len(stops_df)} rows × {len(stops_df.columns)} columns")
print(f"Saved to: {output_path}")
print(f"\nColumns in final dataset:")
for col in stops_df.columns:
    print(f"  - {col}")

In [ ]:
# Display summary statistics
print("\nSummary Statistics:")
print(f"\nParticipants: {stops_df['prolific_id'].nunique()}")
print(f"Tokens per participant: {len(stops_df) / stops_df['prolific_id'].nunique():.1f}")
print(f"\nPhonation distribution:")
print(stops_df['phonation'].value_counts())
print(f"\nPOA distribution:")
print(stops_df['poa'].value_counts())
print(f"\nVOT range: {stops_df['vot'].min():.1f} - {stops_df['vot'].max():.1f} ms")
print(f"F0 range: {stops_df['f0'].min():.0f} - {stops_df['f0'].max():.0f} Hz")
print(f"\nAge range: {stops_df['age'].min():.0f} - {stops_df['age'].max():.0f} years")
print(f"Gender distribution:")
print(stops_df.groupby('prolific_id')['gender'].first().value_counts())